In [1]:
import harpy as hp

Vitessce currently supports Zarr v2, while SpatialData objects are increasingly written in Zarr v3. To make those SpatialData Zarr v3 objects usable in Vitessce, we wrote helper functions that convert the relevant image, segmentation, and table outputs to a Vitessce-compatible Zarr v2 layout.

In [2]:
from importlib.metadata import version

zarr_major_version = int(version("zarr").split(".")[0])
assert zarr_major_version == 3, (
    f"Expected zarr v3 to be installed for this conversion workflow, found zarr v{zarr_major_version}."
)

print(
    f"Using zarr v{zarr_major_version} for the SpatialData to Vitessce conversion workflow."
)

Using zarr v3 for the SpatialData to Vitessce conversion workflow.


Create the environment for this notebook with:

```bash
uv venv .venv_harpy --python 3.12
source .venv_harpy/bin/activate

uv pip install 'harpy-analysis[extra] @ git+https://github.com/saeyslab/harpy.git@main'
uv pip install "harpy_vitessce @ git+https://github.com/vibspatial/harpy_vitessce.git@main"
uv pip install jupyter
```

In [3]:
sdata = hp.datasets.xenium_human_ovarian_cancer(
    subset=True,
    processed=True,
)

no parent found for <ome_zarr.reader.Label object at 0x12d2b9310>: None
no parent found for <ome_zarr.reader.Label object at 0x12d307200>: None
no parent found for <ome_zarr.reader.Label object at 0x12d398f50>: None
/Users/arne.defauw/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


Next, we write the microscopy image and the segmentation mask to OME-Zarr. These files provide the image and label layers that Vitessce will render in the spatial view.

In [4]:
from spatialdata.transformations import get_transformation

get_transformation(sdata["clahe"], get_all=True)

{'global': Sequence 
     Translation (c, y, x)
         [    0. 18000. 10000.]
     Identity ,
 'global_micron': Sequence 
     Translation (c, y, x)
         [    0. 18000. 10000.]
     Sequence 
         Identity 
         Scale (c, y, x)
             [1.     0.2125 0.2125]}

In [5]:
hp.im.get_dataarray(sdata, layer="clahe").c.data

array(['DAPI', 'ATP1A1/CD45/E-Cadherin', '18S', 'AlphaSMA/Vimentin'],
      dtype='<U22')

In [6]:
from pathlib import Path

import harpy as hp
import harpy_vitessce as hpv

BASE_DIR = Path("/Users/arne.defauw/VIB/DATA/test_data/vitessce")

output_path_image = BASE_DIR / "image.ome.zarr"
output_path_mask = BASE_DIR / "mask.ome.zarr"

# NOTE: we choose not to write transformations to the ome.zarr, but we will make vitessce take care of this at rendering time.
hpv.data_utils.xarray_to_ome_zarr(
    tree_or_da=sdata["clahe"],
    output_path=output_path_image,
    channel_names=hp.im.get_dataarray(sdata, layer="clahe").c.data,
    chunks=(1, 2048, 2048),
    scale_factors=[
        2,
        2,
        2,
        2,
    ],  # will be ignored, because sdata["clahe"] is already a datatree
    zarr_format=2,
)

hpv.data_utils.xarray_to_ome_zarr(
    tree_or_da=sdata["nucleus_segmentation_mask"],
    output_path=output_path_mask,
    channel_names=["segmentation_mask"],
    chunks=(2048, 2048),
    scale_factors=[
        2,
        2,
        2,
        2,
    ],  # will be ignored, because sdata["segmentation_mask"] is already a datatree
    zarr_format=2,
)

2026-03-09 08:50:09.253 | WARNING  | harpy_vitessce.data_utils._ome:xarray_to_ome_zarr:132 - scale factors ignored if DataTree


Next, we want to write the result to a chunked AnnData Zarr store on disk. SpatialData can store AnnData tables, but in this workflow we write the AnnData object directly so we can control the chunked on-disk format explicitly.

Vitessce also works well with sparse arrays. In this workflow, however, we already preprocess the AnnData table, so the result can no longer be represented efficiently as a sparse matrix.
`harpy_vitessce.data_utils.normalize_array` prepares dense or sparse numeric arrays before writing them. It converts sparse matrices to CSC format, which is the preferred layout for sparse data in this context. It also downcasts integer-like values to the smallest safe integer dtype to reduce storage size. For floating-point data, it only converts `float64` to `float32` when that conversion is lossless; otherwise it keeps the original dtype.

In [7]:
import anndata as ad
from spatialdata.models import TableModel

ad.settings.zarr_write_format = (
    2  # need to write to zarr 2 to be able to be compatible with vitessce
)

VAR_CHUNK_SIZE = 10

output_path_adata = BASE_DIR / "adata.zarr"

adata_annotated = sdata["table_transcriptomics_preprocessed"]

instance_key = adata_annotated.uns[TableModel.ATTRS_KEY][TableModel.INSTANCE_KEY]

adata_annotated.obs_names = adata_annotated.obs[instance_key].astype(
    str
)  # must match mask labels
adata_annotated.obs_names.name = "obs_id_vitessce"  # avoid name conflict with obs column  # !!! Index of the AnnData table must match the ID in the segmentation mask

adata_annotated.obsm["spatial_micron"] = (
    adata_annotated.obsm["spatial"] * 0.2125
)  # microns per pixel for this experiment was 0.2125

adata_annotated.write_zarr(
    output_path_adata, chunks=[adata_annotated.shape[0], VAR_CHUNK_SIZE]
)

/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy/lib/python3.12/site-packages/anndata/_io/zarr.py:44: UserWarning: Writing zarr v2 data will no longer be the default in the next minor release. v3 data will be written by default. If you are explicitly setting this configuration, consider migrating to the zarr v3 file format.
  f = open_write_group(store)


In [8]:
adata_annotated.obs.head()

,cell_ID,fov_labels,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_2_genes,pct_counts_in_top_5_genes,n_counts,shapeSize,leiden
obs_id_vitessce,,,,,,,,,,,
1620,1620,nucleus_segmentation_mask,113,4.736198,145,4.983607,8.275862,16.551724,145,1494.0,7
1684,1684,nucleus_segmentation_mask,17,2.890372,18,2.944439,16.666667,33.333333,18,565.0,2
1780,1780,nucleus_segmentation_mask,62,4.143135,65,4.189655,6.153846,12.307692,65,898.0,7
1812,1812,nucleus_segmentation_mask,94,4.553877,113,4.736198,7.079646,14.159292,113,1275.0,2
1844,1844,nucleus_segmentation_mask,106,4.672829,137,4.927254,10.218978,19.708029,137,1452.0,7
